In [ ]:
# ============================================
# NOTEBOOK 5: STATISTICAL ANALYSIS
# ============================================

# %% [markdown]
# # Statistical Analysis & Hypothesis Testing
# 
# Objectives:
# - Test key hypotheses
# - Analyze experience impact
# - Identify significant patterns

# %% Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# %% Load Data
df = pd.read_csv('../data/integrated_data.csv')
instructor_metrics = pd.read_csv('../outputs/tables/instructor_metrics.csv')
course_metrics = pd.read_csv('../outputs/tables/course_metrics_full.csv')

print("✅ Data loaded successfully!")

# %% HYPOTHESIS 1: Experience Impact on Teacher Rating
print("\n" + "=" * 60)
print("HYPOTHESIS 1: EXPERIENCE IMPACT ON RATINGS")
print("=" * 60)
print("\nH0: No correlation between years of experience and teacher rating")
print("H1: Positive correlation exists")

# Correlation test
corr_exp_teacher, p_value_1 = stats.pearsonr(
    df['YearsOfExperience'].dropna(),
    df['TeacherRating'].dropna()
)

print(f"\n📊 Results:")
print(f"Correlation Coefficient: {corr_exp_teacher:.4f}")
print(f"P-Value: {p_value_1:.4e}")
print(f"Sample Size: {len(df)}")

if p_value_1 < 0.05:
    print("\n✅ REJECT H0: Significant correlation exists")
    if corr_exp_teacher > 0:
        print(f"   Positive correlation: More experience → Higher ratings")
    else:
        print(f"   Negative correlation: More experience → Lower ratings")
else:
    print("\n❌ FAIL TO REJECT H0: No significant correlation")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot with regression
axes[0].scatter(df['YearsOfExperience'], df['TeacherRating'], 
                alpha=0.5, s=30, color='steelblue')
z = np.polyfit(df['YearsOfExperience'], df['TeacherRating'], 1)
p = np.poly1d(z)
axes[0].plot(df['YearsOfExperience'], p(df['YearsOfExperience']), 
             "r--", linewidth=2, label=f'r = {corr_exp_teacher:.3f}')
axes[0].set_xlabel('Years of Experience', fontsize=12)
axes[0].set_ylabel('Teacher Rating', fontsize=12)
axes[0].set_title('Experience vs Teacher Rating', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot by experience category
df['ExperienceCategory'] = pd.cut(
    df['YearsOfExperience'],
    bins=[0, 3, 7, 15, 100],
    labels=['Beginner\n(0-3)', 'Intermediate\n(3-7)', 
            'Experienced\n(7-15)', 'Veteran\n(15+)']
)

df.boxplot(column='TeacherRating', by='ExperienceCategory', ax=axes[1])
axes[1].set_xlabel('Experience Category', fontsize=12)
axes[1].set_ylabel('Teacher Rating', fontsize=12)
axes[1].set_title('Rating Distribution by Experience Level', fontsize=14, fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis1_experience_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% HYPOTHESIS 2: Gender Impact on Performance
print("\n" + "=" * 60)
print("HYPOTHESIS 2: GENDER DIFFERENCES IN RATINGS")
print("=" * 60)
print("\nH0: No difference in ratings between male and female instructors")
print("H1: Significant difference exists")

male_ratings = instructor_metrics[instructor_metrics['Gender'] == 'Male']['TeacherRating']
female_ratings = instructor_metrics[instructor_metrics['Gender'] == 'Female']['TeacherRating']

# T-test
t_stat, p_value_2 = stats.ttest_ind(male_ratings, female_ratings)

print(f"\n📊 Results:")
print(f"Male Mean Rating: {male_ratings.mean():.4f} (n={len(male_ratings)})")
print(f"Female Mean Rating: {female_ratings.mean():.4f} (n={len(female_ratings)})")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value_2:.4f}")

if p_value_2 < 0.05:
    print("\n✅ REJECT H0: Significant difference exists")
else:
    print("\n❌ FAIL TO REJECT H0: No significant difference")

# Visualization
plt.figure(figsize=(12, 6))


plt.subplot(1, 2, 1)
gender_data = [male_ratings, female_ratings]
plt.boxplot(gender_data, labels=['Male', 'Female'])
plt.ylabel('Teacher Rating', fontsize=12)
plt.title('Rating Distribution by Gender', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
gender_means = [male_ratings.mean(), female_ratings.mean()]
gender_stds = [male_ratings.std(), female_ratings.std()]
plt.bar(['Male', 'Female'], gender_means, yerr=gender_stds, 
        capsize=10, color=['steelblue', 'coral'], alpha=0.7)
plt.ylabel('Mean Teacher Rating', fontsize=12)
plt.title('Average Rating by Gender (with std)', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis2_gender_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% HYPOTHESIS 3: Course Level Impact on Quality
print("\n" + "=" * 60)
print("HYPOTHESIS 3: COURSE LEVEL IMPACT ON RATINGS")
print("=" * 60)
print("\nH0: No difference in ratings across course levels")
print("H1: Significant difference exists")

# Get ratings by level
level_groups = [
    course_metrics[course_metrics['CourseLevel'] == level]['CourseRating'].values
    for level in course_metrics['CourseLevel'].unique()
]

# ANOVA
f_stat_3, p_value_3 = stats.f_oneway(*level_groups)

print(f"\n📊 Results:")
for level in course_metrics['CourseLevel'].unique():
    level_mean = course_metrics[course_metrics['CourseLevel'] == level]['CourseRating'].mean()
    level_count = len(course_metrics[course_metrics['CourseLevel'] == level])
    print(f"{level}: Mean = {level_mean:.4f} (n={level_count})")

print(f"\nF-Statistic: {f_stat_3:.4f}")
print(f"P-Value: {p_value_3:.4e}")

if p_value_3 < 0.05:
    print("\n✅ REJECT H0: Significant differences exist between levels")
else:
    print("\n❌ FAIL TO REJECT H0: No significant differences")

# Visualization
plt.figure(figsize=(12, 6))
course_metrics.boxplot(column='CourseRating', by='CourseLevel', figsize=(12, 6))
plt.suptitle('')
plt.xlabel('Course Level', fontsize=12)
plt.ylabel('Course Rating', fontsize=12)
plt.title('Course Rating Distribution by Level', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis3_level_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% HYPOTHESIS 4: Instructor Quality Impact on Enrollments
print("\n" + "=" * 60)
print("HYPOTHESIS 4: INSTRUCTOR RATING IMPACT ON ENROLLMENTS")
print("=" * 60)
print("\nH0: No correlation between instructor rating and enrollments")
print("H1: Positive correlation exists")

corr_rating_enroll, p_value_4 = stats.pearsonr(
    instructor_metrics['TeacherRating'],
    instructor_metrics['TotalEnrollments']
)

print(f"\n📊 Results:")
print(f"Correlation Coefficient: {corr_rating_enroll:.4f}")
print(f"P-Value: {p_value_4:.4e}")

if p_value_4 < 0.05:
    print("\n✅ REJECT H0: Significant correlation exists")
else:
    print("\n❌ FAIL TO REJECT H0: No significant correlation")

# Visualization
plt.figure(figsize=(12, 8))
plt.scatter(instructor_metrics['TeacherRating'], 
            instructor_metrics['TotalEnrollments'],
            alpha=0.6, s=100, color='purple', edgecolors='black', linewidth=0.5)

z = np.polyfit(instructor_metrics['TeacherRating'], 
               instructor_metrics['TotalEnrollments'], 1)
p = np.poly1d(z)
plt.plot(instructor_metrics['TeacherRating'], 
         p(instructor_metrics['TeacherRating']), 
         "r--", linewidth=2, label=f'r = {corr_rating_enroll:.3f}')

plt.xlabel('Teacher Rating', fontsize=12)
plt.ylabel('Total Enrollments', fontsize=12)
plt.title('Instructor Rating vs Total Enrollments', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis4_enrollment_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Experience Diminishing Returns Analysis
print("\n" + "=" * 60)
print("DIMINISHING RETURNS ANALYSIS")
print("=" * 60)


# Calculate rating change per experience year
exp_rating_avg = df.groupby('YearsOfExperience')['TeacherRating'].mean().reset_index()

# Calculate first derivative (rate of change)
exp_rating_avg['RatingChange'] = exp_rating_avg['TeacherRating'].diff()

print("\n📊 Average Rating by Years of Experience:")
print(exp_rating_avg.head(15))

# Find inflection point
max_benefit = exp_rating_avg['RatingChange'].idxmax()
if pd.notna(max_benefit):
    optimal_exp = exp_rating_avg.loc[max_benefit, 'YearsOfExperience']
    print(f"\n✅ Maximum rating increase occurs around {optimal_exp} years")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average rating by experience
axes[0].plot(exp_rating_avg['YearsOfExperience'], 
             exp_rating_avg['TeacherRating'], 
             marker='o', linewidth=2, markersize=6, color='steelblue')
axes[0].set_xlabel('Years of Experience', fontsize=12)
axes[0].set_ylabel('Average Teacher Rating', fontsize=12)
axes[0].set_title('Average Rating by Years of Experience', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Rate of change
axes[1].plot(exp_rating_avg['YearsOfExperience'][1:], 
             exp_rating_avg['RatingChange'][1:], 
             marker='o', linewidth=2, markersize=6, color='coral')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Years of Experience', fontsize=12)
axes[1].set_ylabel('Rating Change (per year)', fontsize=12)
axes[1].set_title('Marginal Rating Improvement by Experience', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/diminishing_returns.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Multiple Regression Analysis
print("\n" + "=" * 60)
print("MULTIPLE REGRESSION: PREDICTING TEACHER RATING")
print("=" * 60)

# Prepare data
regression_data = instructor_metrics[['YearsOfExperience', 'Age', 
                                       'TotalEnrollments', 'CoursesTaught',
                                       'TeacherRating']].dropna()

X = regression_data[['YearsOfExperience', 'Age', 'TotalEnrollments', 'CoursesTaught']]
y = regression_data['TeacherRating']

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit model
model = LinearRegression()
model.fit(X_scaled, y)

# Results
print(f"\n📊 Model Performance:")
print(f"R² Score: {model.score(X_scaled, y):.4f}")
print(f"\nCoefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.abs(model.coef_)
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color='teal')
plt.xlabel('Absolute Coefficient (Standardized)', fontsize=12)
plt.title('Feature Importance in Predicting Teacher Rating', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Statistical Summary Report
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY REPORT")
print("=" * 60)

summary_report = pd.DataFrame({
    'Hypothesis': [
        'H1: Experience → Rating',
        'H2: Gender Differences',
        'H3: Level Impact',
        'H4: Rating → Enrollment'
    ],
    'Test': [
        'Pearson Correlation',
        'Independent T-Test',
        'One-Way ANOVA',
        'Pearson Correlation'
    ],
    'Statistic': [
        f'r = {corr_exp_teacher:.4f}',
        f't = {t_stat:.4f}',
        f'F = {f_stat_3:.4f}',
        f'r = {corr_rating_enroll:.4f}'
    ],
    'P-Value': [
        f'{p_value_1:.4e}',f'{p_value_2:.4f}',
        f'{p_value_3:.4e}',
        f'{p_value_4:.4e}'
    ],
    'Result': [
        'Reject H0' if p_value_1 < 0.05 else 'Fail to Reject',
        'Reject H0' if p_value_2 < 0.05 else 'Fail to Reject',
        'Reject H0' if p_value_3 < 0.05 else 'Fail to Reject',
        'Reject H0' if p_value_4 < 0.05 else 'Fail to Reject'
    ]
})

print("\n", summary_report.to_string(index=False))

# Save report
summary_report.to_csv('../outputs/tables/statistical_tests_summary.csv', index=False)

# %% [markdown]
# ## ✅ Statistical Analysis Complete
# 
# Key Findings:
# 1. Experience shows [positive/negative/no] correlation with ratings
# 2. Gender [shows/doesn't show] significant differences
# 3. Course levels [have/don't have] significant quality differences
# 4. High-rated instructors [attract/don't attract] more enrollments
# 5. Optimal experience threshold: ~X years

print("\n" + "=" * 60)
print("✅ NOTEBOOK 5 COMPLETED SUCCESSFULLY!")
print("=" * 60)

